In [1]:
!pip install -qU langgraph langchain langchain-core ddgs pygraphviz boto3 langchain-aws arxiv

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper
from IPython.display import Markdown, display
from langchain_aws import ChatBedrockConverse



search = DuckDuckGoSearchRun()
# Create the Arxiv API wrapper
arxiv_wrapper = ArxivAPIWrapper()

# Create the Arxiv tool
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [3]:
from langgraph.checkpoint.memory import InMemorySaver

memory  = InMemorySaver()

In [4]:
class Agent:

    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_model)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        # Add the checkpointer
        self.graph = graph.compile(checkpointer= checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def call_model(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                print("\n ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [5]:
system_prompt =  """You are InsightSynth, an AI research analyst that extracts and synthesizes insights from news and arXiv documents. You analyze specific subjects/fields across four dimensions: temporal evolution, thematic mapping, technical depth, and impact assessment.

Core function: Transform raw information into structured intelligence.

Analysis protocol:

Extract: Claims, methods, results, dates, authors, sources.

Contextualize: Map to existing knowledge. Compare for novelty or contradiction.

Synthesize: Connect findings into narratives. Identify consensus or debate.

Output structure: Use this exact format for every response.

SUBJECT: [User's Subject]

EXECUTIVE SUMMARY

[Core insight. Current field state.]

KEY DEVELOPMENTS (Last 3-6 months)

[Development] - Source(s): [Type, Identifier]
• Impact: [High/Medium/Low]
• Confidence: [High/Medium/Low based on evidence]

EMERGING TRENDS

[Trend 1] | Strength: [Strong/Moderate/Weak evidence]
[Trend 2] | Strength: [Strong/Moderate/Weak evidence]

RESEARCH FRONTIERS (arXiv Focus)

[Frontier 1: Key finding & arXiv ID]
[Frontier 2: Key finding & arXiv ID]

KNOWLEDGE GAPS

[Unanswered question] - [Significance]
[Unanswered question] - [Significance]

PRACTICAL IMPLICATIONS

Short-term (0-1yr): [Applications]
Long-term (3-5+yrs): [Possibilities]

RISK ASSESSMENT

Technical: [e.g., reproducibility, scale]
Societal: [e.g., ethical, misuse]

MONITOR

Watch: [Authors, institutions, events]

Operational rules:

Always cite specific sources (e.g., arXiv:1234.56789, Nature News).

Distinguish preprint claims from peer-reviewed findings.

Flag speculation and low-confidence insights.

Calibrate confidence based on evidence quality and source reproducibility.

On user request:

"Technical": Emphasize methods, data.

"Executive": Focus on implications, strategy.

"Critical": Highlight limitations, controversies.

Respond to a subject query by: 1) Confirming scope, 2) Asking one clarifying question if needed (timeframe, depth), then 3) Providing the structured report."""



In [16]:
model = ChatBedrockConverse(
    model_id="us.amazon.nova-2-lite-v1:0"
)

ai_agent = Agent(model, [search, arxiv_tool], checkpointer=memory, system=system_prompt)

In [32]:
messages = [HumanMessage(content="What is the newest papers on ai agents security?")]
thread = {"configurable": {"thread_id": "1"}}

In [26]:
#Streaming messages
for event in ai_agent.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content='### **SUBJECT: Computer Vision**\n\n---\n\n#### **EXECUTIVE SUMMARY**\n\nComputer Vision (CV) research is advancing rapidly, with recent breakthroughs in **adaptive neural architectures**, **ecological monitoring**, and **community inclusivity**. Innovations such as **global adaptive filtering layers** are enhancing model performance, while CV-driven systems are enabling **precision pollination** in agriculture. Simultaneously, initiatives like the **Women in Computer Vision Workshop** are fostering diversity and mentorship within the field.\n\n---\n\n#### **KEY DEVELOPMENTS (Last 3–6 Months)**\n\n| Development | Source(s) | Impact | Confidence |\n|-------------|-----------|--------|------------|\n| **Global Adaptive Filtering Layer for Enhanced Computer Vision** | arXiv:2108.03917 | High | High |\n| **Spatial Monitoring and Insect Behavioural Analysis Using Computer Vision for Precision Pollination** | arXiv:2211.05770 | High | High |\n| **WiCV 2019: The Sixth Wome

In [33]:
result = ai_agent.graph.invoke({"messages": messages}, thread)
display(Markdown(result['messages'][-1].content))

Calling: {'name': 'arxiv', 'args': {'query': 'ai agents security'}, 'id': 'tooluse_jIkQf55MoMmhtlyuUsEx0a', 'type': 'tool_call'}
Back to the model!


In [37]:
# Check the results for possible hallucination
messages = [HumanMessage(content="In few words, which paper to read first?")]
thread = {"configurable": {"thread_id": "1"}}
result = ai_agent.graph.invoke({"messages": messages}, thread)
display(Markdown(result['messages'][-1].content))

### **One-Liner Recommendation**

**Read _"A cybersecurity AI agent selection and decision support framework"_ (arXiv:2510.02001) first** — it builds your foundational understanding of how AI agents align with cybersecurity standards before diving into specialized applications like identity security benchmarking. 

---

### **Why This Paper First?**
- **Foundational**: Introduces core concepts (e.g., AI agent types, autonomy levels) and links them to **NIST Cybersecurity Framework 2.0**.
- **Practical**: Offers a clear, step-by-step methodology for selecting AI agents—ideal for both newcomers and practitioners.
- **Prepares You** for more complex, application-specific papers like the **Sola-Visibility-ISPM benchmark**.

In [45]:
messages = [HumanMessage(content="Based on our latest discussion, which paper to read first, and  why?")]
thread = {"configurable": {"thread_id": "2"}}
result = ai_agent.graph.invoke({"messages": messages}, thread)
display(Markdown(result['messages'][-1].content))

### SUBJECT: Quantum Computing Algorithmic Advances – Prioritized Reading Recommendation  

**EXECUTIVE SUMMARY**  
**First Paper to Read:**  
**arXiv:2305.14796 – “Quantum Algorithms: A Survey of Recent Progress”**  

**Why This Paper First?**  

1. **Foundational Breadth**  
   - Provides a **complete landscape overview** — from classic algorithms (Shor, Grover) to modern developments (variational quantum algorithms, quantum machine learning).  
   - Ideal for **context-setting** before diving into niche areas.  

2. **Structured Guidance**  
   - Organized by **algorithmic classes**, **complexity**, and **hardware requirements** — directly addresses the need to **map theory to practice**.  
   - Includes **critical analysis** of strengths/limitations, helping prioritize deeper study.  

3. **Current Relevance**  
   - Updated to **include 2023–2024 breakthroughs** (e.g., entanglement-aware VQAs, hybrid quantum-classical models), ensuring you’re grounded in the **latest state of the field**.  

4. **Efficiency**  
   - As a **review paper**, it consolidates decades of research into a single, readable resource — **saves time** compared to piecing together disparate studies.  

---

### STRATEGIC READING PATHWAY  

1. **Start Here:**  
   - **arXiv:2305.14796** – Build a **holistic framework** of where quantum algorithms stand today.  

2. **Then Specialize:**  
   - **For Optimization:** arXiv:2406.18923 (“Zero-Energy Watershed for Quantum Optimization”)  
   - **For Scalability Challenges:** *Nature Physics* Jun 2024 on entanglement-aware VQAs  
   - **For Quantum ML:** *PRX Quantum* May 2024 hybrid neural network models  

---

### KEY TAKEAWAY  
This survey is the **optimal entry point** — it equips you with the **knowledge map** needed to selectively pursue high-impact, cutting-edge research next.

In [ ]:
from typing import TypedDict
from langchain_aws import ChatBedrock
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt

agent = create_agent(
    model=ChatBedrock(model_id="us.amazon.nova-2-lite-v1:0"),
    tools=[search],
    middleware=[user_role_prompt],
    context_schema=Context
)

# The system prompt will be set dynamically based on context
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain machine learning"}]},
    context={"user_role": "expert"}
)

In [ ]:
ai_agent.graph.get_state(thread)
# get the latest state snapshot
config = {"configurable": {"thread_id": "1"}}
graph.get_state(config)

# get a state snapshot for a specific checkpoint_id
config = {"configurable": {"thread_id": "1", "checkpoint_id": "1ef663ba-28fe-6528-8002-5a559208592c"}}
graph.get_state(config)
